# Quickstart to demonstrate an OCP agent

In [1]:
!pip install deepagents langchain_openai langchain_mcp_adapters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 77.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 126.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.5/832.5 kB 121.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 114.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 117.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 145.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 140.3 MB/s  0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 16.0
    Uninstalling websockets-16.0:
      Successfully uninstalled websockets-16.0━━━━━━━━━━━━━━━━━━━━  3/37 [websockets]
  Attempting uninstall: google-auth╸━━━━━━━━━━━━━━━━━ 21/37 [langsmith]
    Found existing installation: google-auth 2.47.0━━━━━━━━━━━ 21/37 [langsmith]
    Uninstalling google-auth-2.47.0:━━━━━━━━━━━━━━━━━ 21/37 [langsmith]
      Successfully uninstalled 

## Create MCP tool Client

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient  
import os

In [3]:
ocp_mcp_endpoint = os.environ["OCP_MCP_ENDPOINT"]
print(ocp_mcp_endpoint)
mcp_servers = {
        "openshift-mcp": {
            "transport": "http",
            "url": ocp_mcp_endpoint,
        }
    }

print(mcp_servers)

mcp_client = MultiServerMCPClient(mcp_servers)

tools = await mcp_client.get_tools()    

#print(tools)

http://ocp-mcp.apps.cluster-pdxmn.dynamic2.redhatworkshops.io/mcp
{'openshift-mcp': {'transport': 'http', 'url': 'http://ocp-mcp.apps.cluster-pdxmn.dynamic2.redhatworkshops.io/mcp'}}


# Create a connection to our LLM

In [4]:
from langchain_openai import ChatOpenAI

qwen = ChatOpenAI(
    base_url=os.environ["QWEN_ENDPOINT"],
    api_key=os.environ["QWEN_API_KEY"],
    model="qwen3-14b",
)

## let's create a subagent that specialises in OCP stuff

In [5]:
ocp_prompt = """You are an expert on Openshift and Kubernetes. Your job is to gather information about a connected OpenShift cluster. Use the MCP tools at your disposal to accomplish this
"""

In [6]:
from deepagents import AsyncSubAgent

ocp_subagent = AsyncSubAgent(
    name="ocp-agent",
    description="Used to report back on the state and to investigate a connected OCP cluster",
    system_prompt=ocp_prompt,
    tools=tools)

    

# let's create the main agent

In [7]:
master_prompt = """You are a general purpose agent. You have access to a subagent that specialises in Openshift and Kubernetes and is connected to a live Openshift cluster"""

In [8]:
from deepagents import create_deep_agent

subagents = [ocp_subagent]

agent = create_deep_agent(
    model=qwen,
    subagents=subagents,
    system_prompt=master_prompt,
)

# Test the agent

In [ ]:
result = await agent.ainvoke({"messages": [{"role": "user", "content": "List the pods currently running in the ocp-mcp-server namespace"}]})

# Print the agent's response
print(result["messages"][-1].content)